# Run 000 — Exploratory Data Analysis

**Amazon ML Challenge 2024 — entity value extraction from product images.**

Predict `"<number> <unit>"` for a given (image, `entity_name`) pair.
Scored by **F1 with exact string match**.

This notebook runs **before any training**. CPU-only, a couple of minutes. Its
job is to answer the questions that decide what we build:

1. Distribution over `entity_name` — the label distribution
2. Unit mix within each entity
3. Empty / missing `entity_value` rate
4. Numeric value distribution per entity (log scale, outliers flagged)
5. `group_id` cardinality and long-tail shape
6. **Value string format audit** — the one that predicts post-processing work
7. Duplicate `image_link` count

All logic lives in `src/amlc24/data/eda.py`; this notebook only calls it and
displays the result.

> **Attach the competition dataset first** (`+ Add Input` in the sidebar).
> The first cell checks for it and stops with a clear message if it is missing.

## 1. Setup

In [ ]:
# Locate the repo and put its `src/` on sys.path.
#
# Layouts supported, in priority order:
#   1. repo git-cloned into the session -> /kaggle/working/Amazon-ML-24/src
#   2. repo uploaded as a Kaggle Dataset -> /kaggle/input/<slug>/.../src
#   3. running locally from the repo itself
#
# No dataset slug is hardcoded. paths.py finds the competition CSVs wherever
# they are mounted, including the official nested layout
# "<slug>/student_resource 3/dataset/".
import sys, glob, subprocess
from pathlib import Path

REPO_URL = "https://github.com/MurtuzaShaikh26/Amazon-ML-24.git"
CLONE_DIR = Path("/kaggle/working/Amazon-ML-24")


def find_src():
    candidates = [CLONE_DIR / "src"]
    for root in sorted(glob.glob("/kaggle/input/*/")):
        candidates.append(Path(root) / "src")
        candidates.extend(Path(p) for p in glob.glob(root + "*/src"))
    candidates.extend(Path(p) for p in sorted(glob.glob("/kaggle/working/*/src")))
    candidates.extend([Path.cwd() / "src", Path.cwd().parent / "src"])
    for c in candidates:
        if (c / "amlc24" / "__init__.py").exists():
            return c.resolve()
    return None


# Always run the latest GitHub code. A Kaggle Dataset linked to GitHub never
# syncs on push, so an attached repo dataset is a stale snapshot; the fresh
# clone must win. The dataset copy is only a fallback when there is no internet.
if CLONE_DIR.exists():
    print("Existing clone found; pulling latest...")
    subprocess.run(["git", "-C", str(CLONE_DIR), "fetch", "--depth", "1", "origin", "main"],
                   capture_output=True, text=True)
    subprocess.run(["git", "-C", str(CLONE_DIR), "reset", "--hard", "origin/main"],
                   capture_output=True, text=True)
else:
    print("Cloning repo...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(CLONE_DIR)],
                   capture_output=True, text=True)

if (CLONE_DIR / "src" / "amlc24" / "__init__.py").exists():
    SRC = (CLONE_DIR / "src").resolve()
    commit = subprocess.run(["git", "-C", str(CLONE_DIR), "log", "-1", "--format=%h %s"],
                            capture_output=True, text=True).stdout.strip()
    print("Running GitHub commit:", commit)
else:
    SRC = find_src()
    print("WARNING: clone failed (internet off?). Using fallback copy:", SRC)
assert SRC is not None, "Could not find the amlc24 package; check the clone above."
sys.path.insert(0, str(SRC))

# Drop any already-imported copy so a pull takes effect without a kernel restart.
for name in [m for m in list(sys.modules) if m == "amlc24" or m.startswith("amlc24.")]:
    del sys.modules[name]

import amlc24
from amlc24.logging_utils import setup_logging
from amlc24.paths import describe

setup_logging()
print()
print("amlc24", amlc24.__version__, " src:", SRC)

info = describe()
for k, v in info.items():
    print(f"  {k:20s} {v}")

if not info.get("train_csv_found"):
    print()
    print("*** train.csv NOT FOUND ***")
    print("Attach the competition dataset: '+ Add Input' in the notebook sidebar,")
    print("search for the Amazon ML Challenge 2024 dataset, add it, then re-run.")
    print("Currently mounted inputs:", info.get("mounted_inputs"))
    raise SystemExit("Competition dataset not attached.")

print()
print("Data located. Ready.")

## 2. Run the profile

`run_eda()` loads `train.csv`, computes every table, saves them to
`results/eda/`, and renders the charts.

In [ ]:
from amlc24.pipeline.run_eda import run_eda

result = run_eda(plots=True)
profile = result["profile"]

print()
print(f"Profiled {result['n_rows']:,} rows")
print("Tables:", ", ".join(profile))
print("Saved to:", result["out_dir"])

## 3. Label distribution over `entity_name`

The primary label distribution: how many rows of each entity type, and how many
units each one permits.

In [ ]:
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)

display(profile["entity_distribution"])

In [ ]:
from amlc24.data.eda import plot_entity_distribution
from amlc24.data.load import load_train

df = load_train()
plot_entity_distribution(df);

## 4. Unit distribution within each entity

Which units actually appear per entity, and whether they are in the allowed list
from `constants.py`. Units outside that list are **invalid predictions** — so
any appearing here in the *labels* are important to know about.

In [ ]:
display(profile["unit_distribution"])

### Cross-tab (entity x unit)

In [ ]:
display(profile["unit_crosstab"])

In [ ]:
from amlc24.data.eda import plot_unit_distribution
plot_unit_distribution(df);

### Allowed units per entity, from the dataset's `constants.py`

In [ ]:
display(profile["allowed_units"])

## 5. Empty / missing `entity_value`

Empty labels are **not** noise — the metric scores a correct empty prediction as
a true negative. The empty rate is the share of the eval set we can score on by
correctly abstaining.

In [ ]:
display(profile["empty_rate"])

In [ ]:
from amlc24.data.eda import plot_empty_rate
plot_empty_rate(df);

## 6. Numeric value distribution per entity

Min / median / max plus robust (MAD-based) outlier counts. These distributions
are heavy-tailed, so a standard-deviation outlier rule would be useless.

In [ ]:
display(profile["value_stats"])

In [ ]:
from amlc24.data.eda import plot_value_histograms
plot_value_histograms(df);

## 7. `group_id` distribution

How many distinct product categories, how big the largest is, how long the tail.

In [ ]:
display(profile["group_distribution"])
display(profile["group_top"])

In [ ]:
from amlc24.data.eda import plot_group_tail
plot_group_tail(df);

## 8. Value string format audit — read this one carefully

Because the metric is **exact string match**, formatting is worth as much as the
number. This table is the specification for
`postprocess/normalize.py::format_number`:

* `trailing_dot_zero` says whether predictions must carry `.0`
* `thousands_separator` says whether commas ever appear
* `range_value` sizes how often the `range_rule` fires
* `unit_outside_allowed_list` sizes the alias table's job

The current defaults (`number_format: float`, `range_rule: bracket`) were set
from this audit, not assumed.

In [ ]:
display(profile["format_audit"])

### Concrete examples of each awkward format

In [ ]:
display(profile["format_examples"])

## 9. Duplicate images

One product photo can carry several entities, so the number of images to
download is below the row count.

In [ ]:
display(profile["image_duplication"])
display(profile["entities_per_image"].head(15))

## 10. Create / verify the frozen 5k eval split

The split is committed in the repo, so this **loads and verifies** it rather
than regenerating. A mismatch raises `SplitMismatch` — that would mean
`train.csv` changed, and every committed score would be suspect.

The table below is the stratification check: `entity_name` proportions must
match across the full file, the eval split, and the training subset to within
1 percentage point.

In [ ]:
from amlc24.config import load_config
from amlc24.data.splits import get_or_create_split

cfg = load_config("run001_qwen2vl_8bit_10k")
split = get_or_create_split(
    df,
    seed=int(cfg.seed),
    eval_size=int(cfg.data.eval_size),
    train_size=int(cfg.data.train_size),
    path=cfg.data.split_file,
)

print(f"eval_5k       {len(split['eval_5k']):,} rows  (FROZEN, immutable)")
print(f"train_subset  {len(split.get('train_subset', [])):,} rows")
print(f"pool          {len(split.get('pool', [])):,} rows")
overlap = set(split["eval_5k"]) & set(split.get("train_subset", []))
print(f"overlap       {len(overlap)}")

table = split.get("proportion_table")
if table is not None:
    display(pd.DataFrame(table) if not hasattr(table, "columns") else table)

## 11. Observations

<!-- Fill this in after reading the tables above. -->